# Inhibitory Patch-seq ETL

Register VISp inhibitory Patch-seq cells, morphology features, T-type mappings, and measured MET-type memberships in CCM format.

In [1]:
from pathlib import Path

import pandas as pd
import polars as pl
import pyarrow as pa
from deltalake import write_deltalake

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import write_models
from connects_common_connectivity.io.arrow_utils import build_cell_feature_matrix_schema
from connects_common_connectivity.io.write_utils import walk_ancestors
from connects_common_connectivity.models import (
    CellFeatureDefinition,
    CellFeatureMatrix,
    CellFeatureSet,
    CellToClusterMapping,
    ClusterMembership,
    DataItem,
    DataItemDataSetAssociation,
    DataSet,
    MappingSet,
    Modality,
)

In [2]:
REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
DATA_ROOT = REPO_ROOT / "data/INH Patch-seq"
OUTPUT_ROOT = Path(get_settings().output_root)

PROJECT_ID = "visp_patchseq"
DATASET_ID = "visp_inh_patchseq"
FEATURE_SET_ID = "inh_visp_morph_features"
TTYPE_HIERARCHY_ID = "tasic_2018_visp_taxonomy"
METTYPE_HIERARCHY_ID = "visp_met_types_taxonomy"
TTYPE_MAPPING_SET_ID = "visp_inh_patchseq_ttype_mapping"
PUBLICATION = "https://doi.org/10.1016/j.cell.2020.09.057"

TTYPE_PATH = DATA_ROOT / "patchseq_tx_cell_ttype_labels.csv"
METTYPE_PATH = DATA_ROOT / "visp_met_cell_assignments_text_names.csv"
DEFINITIONS_PATH = DATA_ROOT / "inh_visp_patchseq_morph_feature_definitions.csv"
FEATURES_PATH = DATA_ROOT / "inh_ivscc_features_wide_unnormalized.csv"
for path in (TTYPE_PATH, METTYPE_PATH, DEFINITIONS_PATH, FEATURES_PATH):
    assert path.is_file(), f"Missing local input: {path}"

## Dataset and DataItems

In [3]:
ttype_source = pd.read_csv(TTYPE_PATH, index_col="spec_id_label", dtype={"spec_id_label": str})
met_source = pd.read_csv(METTYPE_PATH, dtype={"specimen_id": str})
feature_source = pd.read_csv(FEATURES_PATH).rename(columns={"specimen_id": "id"})
feature_source["id"] = feature_source["id"].astype(str)

# Register the union now so later payloads and assignments cannot introduce orphan IDs.
registered_ids = (
    set(ttype_source.index.astype(str))
    | set(met_source["specimen_id"].astype(str))
    | set(feature_source["id"])
)
assert registered_ids
dataset = DataSet(
    id=DATASET_ID,
    name="VISp inhibitory Patch-seq cells",
    publication=PUBLICATION,
    modality=Modality.MORPHOLOGY,
    project_id=PROJECT_ID,
)
dataitems = [DataItem(id=cell_id, name=cell_id, project_id=PROJECT_ID) for cell_id in sorted(registered_ids)]
associations = [
    DataItemDataSetAssociation(
        dataitem_id=cell_id, dataset_id=DATASET_ID, project_id=PROJECT_ID
    )
    for cell_id in sorted(registered_ids)
]
print(write_models([dataset], output_root=OUTPUT_ROOT))
print(write_models(dataitems, output_root=OUTPUT_ROOT))
print(write_models(associations, output_root=OUTPUT_ROOT))

WrittenResult(class_name='DataSet', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/dataset'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'visp_patchseq\')',), rows_written=1)
WrittenResult(class_name='DataItem', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/dataitem'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'visp_patchseq\')',), rows_written=2879)
WrittenResult(class_name='DataItemDataSetAssociation', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/dataitem_dataset_association'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."dataset_id" = source."dataset_id" AND target."dataitem_id" = source."dataitem_id" AND target."project_id" IN (\'visp_patchseq

## Morphology features

In [4]:
definition_source = pd.read_csv(DEFINITIONS_PATH)
feature_definitions = []
for row in definition_source.to_dict("records"):
    values = {
        "id": str(row["id"]),
        "description": str(row["description"]),
        "unit": str(row["unit"]),
        "data_type": str(row["data_type"]),
        "project_id": PROJECT_ID,
        "feature_set_id": FEATURE_SET_ID,
    }
    if pd.notna(row.get("range_min")):
        values["range_min"] = float(row["range_min"])
    if pd.notna(row.get("range_max")):
        values["range_max"] = float(row["range_max"])
    feature_definitions.append(CellFeatureDefinition(**values))

feature_ids = [definition.id for definition in feature_definitions]
feature_set = CellFeatureSet(
    id=FEATURE_SET_ID,
    description="Unnormalized morphology features for inhibitory VISp Patch-seq cells.",
    feature_definition_ids=feature_ids,
    extraction_method="IVSCC morphology feature extraction",
    project_id=PROJECT_ID,
)
assert len(feature_definitions) == 46
assert set(feature_source.columns) == {"id", *feature_ids}
print(write_models(feature_definitions, output_root=OUTPUT_ROOT))
print(write_models([feature_set], output_root=OUTPUT_ROOT))

WrittenResult(class_name='CellFeatureDefinition', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/cellfeaturedefinition'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."feature_set_id" = source."feature_set_id" AND target."id" = source."id" AND target."project_id" IN (\'visp_patchseq\') AND target."feature_set_id" IN (\'inh_visp_morph_features\')',), rows_written=46)
WrittenResult(class_name='CellFeatureSet', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/cellfeatureset'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'visp_patchseq\')',), rows_written=1)


In [5]:
# Wide payloads keep one row per cell while metadata tables describe each column.
feature_wide = feature_source[["id", *feature_ids]].copy()
feature_wide.insert(1, "project_id", PROJECT_ID)
feature_wide.insert(2, "feature_set_id", FEATURE_SET_ID)
feature_schema = build_cell_feature_matrix_schema(
    feature_set, feature_definitions, cell_index_column="id"
)
feature_table = pa.Table.from_pandas(feature_wide, schema=feature_schema, preserve_index=False)
feature_path = OUTPUT_ROOT / f"cellfeatures/{FEATURE_SET_ID}"
write_deltalake(
    feature_path,
    feature_table,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}' AND feature_set_id = '{FEATURE_SET_ID}'",
    partition_by=["project_id", "feature_set_id"],
)
feature_matrix = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{FEATURE_SET_ID}",
    feature_set_id=FEATURE_SET_ID,
    parquet_path=feature_path.relative_to(OUTPUT_ROOT).as_posix(),
    cell_index_column="id",
    project_id=PROJECT_ID,
)
print(write_models([feature_matrix], output_root=OUTPUT_ROOT))

WrittenResult(class_name='CellFeatureMatrix', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/cellfeaturematrix'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."feature_set_id" = source."feature_set_id" AND target."id" = source."id" AND target."project_id" IN (\'visp_patchseq\')',), rows_written=1)


## Cell-type assignments

In [6]:
# Assignments target shared hierarchy nodes and therefore require both taxonomies.
cluster_table = pl.read_delta(OUTPUT_ROOT / "cluster")
ttype_clusters = cluster_table.filter(pl.col("hierarchy_id") == TTYPE_HIERARCHY_ID)
mettype_clusters = cluster_table.filter(pl.col("hierarchy_id") == METTYPE_HIERARCHY_ID)
assert not ttype_clusters.is_empty(), "Run the shared T-type taxonomy ETL first."
assert not mettype_clusters.is_empty(), "Run the shared MET-type taxonomy ETL first."
ttype_parent = dict(ttype_clusters.select("id", "parent").iter_rows())
mettype_parent = dict(mettype_clusters.select("id", "parent").iter_rows())

In [7]:
ttype_clean = ttype_source.dropna(subset=["ttype"]).copy()
assert not ttype_clean["ttype"].astype(str).str.contains("ET").any()
unknown_ttypes = set(ttype_clean["ttype"]) - set(ttype_parent)
assert not unknown_ttypes, f"Unknown T-types: {sorted(unknown_ttypes)}"
ttype_mapping_set = MappingSet(
    id=TTYPE_MAPPING_SET_ID,
    name="VISp inhibitory Patch-seq T-type assignments",
    description="Tree mapping of source T-type labels onto the Tasic 2018 VISp taxonomy.",
    method_name="Patch-seq tree mapping",
    source_dataset=DATASET_ID,
    target_hierarchy=TTYPE_HIERARCHY_ID,
    project_id=PROJECT_ID,
)
ttype_mappings = []
for cell_id, leaf in ttype_clean["ttype"].items():
    for cluster_id, is_leaf in walk_ancestors(leaf, ttype_parent):
        ttype_mappings.append(
            CellToClusterMapping(
                id=f"{cell_id}-{cluster_id}-{PROJECT_ID}-{TTYPE_HIERARCHY_ID}",
                mapping_set=TTYPE_MAPPING_SET_ID,
                source_cell=str(cell_id),
                target_cluster=cluster_id,
                project_id=PROJECT_ID,
            )
        )
print(write_models([ttype_mapping_set], output_root=OUTPUT_ROOT))
print(write_models(ttype_mappings, output_root=OUTPUT_ROOT))

WrittenResult(class_name='MappingSet', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/mappingset'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."id" = source."id" AND target."project_id" IN (\'visp_patchseq\')',), rows_written=1)


WrittenResult(class_name='CellToClusterMapping', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/celltoclustermapping'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."mapping_set" = source."mapping_set" AND target."id" = source."id" AND target."project_id" IN (\'visp_patchseq\')',), rows_written=11036)


In [8]:
# Source-observed MET labels are memberships, not predictions or cross-taxonomy mappings.
met_clean = met_source.dropna(subset=["met_type"]).copy()
unknown_mettypes = set(met_clean["met_type"]) - set(mettype_parent)
assert not unknown_mettypes, f"Unknown MET-types: {sorted(unknown_mettypes)}"
memberships = []
for row in met_clean[["specimen_id", "met_type"]].to_dict("records"):
    for cluster_id, is_leaf in walk_ancestors(row["met_type"], mettype_parent):
        memberships.append(
            ClusterMembership(
                item=str(row["specimen_id"]),
                cluster=cluster_id,
                hierarchy_id=METTYPE_HIERARCHY_ID,
                probability=1.0 if is_leaf else None,
                project_id=PROJECT_ID,
            )
        )
print(write_models(memberships, output_root=OUTPUT_ROOT))

WrittenResult(class_name='ClusterMembership', path=PosixPath('/workspaces/ConnectsCommonConnectivity/scratch/ccm_release_202610_v7/clustermembership'), mode='merge_scoped', predicates=('target."project_id" = source."project_id" AND target."hierarchy_id" = source."hierarchy_id" AND target."item" = source."item" AND target."cluster" = source."cluster" AND target."project_id" IN (\'visp_patchseq\') AND target."hierarchy_id" IN (\'visp_met_types_taxonomy\')',), rows_written=1485)


## Verify

In [9]:
# Verify scoped records without assuming this shared project contains only one cohort.
stored_associations = pl.read_delta(OUTPUT_ROOT / "dataitem_dataset_association").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID)
)
stored_features = pl.read_delta(feature_path).filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("feature_set_id") == FEATURE_SET_ID)
)
stored_mappings = pl.read_delta(OUTPUT_ROOT / "celltoclustermapping").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("mapping_set") == TTYPE_MAPPING_SET_ID)
)
stored_memberships = pl.read_delta(OUTPUT_ROOT / "clustermembership").filter(
    (pl.col("project_id") == PROJECT_ID)
    & (pl.col("hierarchy_id") == METTYPE_HIERARCHY_ID)
    & pl.col("item").is_in(sorted(set(met_clean["specimen_id"])))
)
assert set(stored_associations["dataitem_id"]) == registered_ids
assert set(stored_features["id"]) == set(feature_source["id"])
assert stored_mappings.height == len(ttype_mappings)
assert stored_memberships.height == len(memberships)
assert set(stored_mappings["target_cluster"]) <= set(ttype_parent)
assert set(stored_memberships["cluster"]) <= set(mettype_parent)
print("Inhibitory Patch-seq CCM records verified.")

Inhibitory Patch-seq CCM records verified.
